# Module 04: Lab 01
## Visual Reporting and Storytelling
**Author:** Maria Taylor

In [ ]:
from pyspark.sql import SparkSession
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

spark = SparkSession.builder.config('spark.driver.host', 'localhost').appName('LightcastData').getOrCreate()
df = spark.read.option('header', 'true').option('inferSchema', 'true').option('multiLine', 'true').option('escape', '"').csv('/home/ubuntu/data/lightcast_job_postings.csv')
df.printSchema()
df.show(5)

## Section 2: Salary Distribution by Employment Type

In [ ]:
df_salary = df.filter((df.SALARY_FROM.isNotNull()) & (df.SALARY_FROM > 0)).select('EMPLOYMENT_TYPE_NAME', 'SALARY_FROM').toPandas()
fig = px.box(df_salary, x='EMPLOYMENT_TYPE_NAME', y='SALARY_FROM', title='Salary Distribution by Employment Type', color='EMPLOYMENT_TYPE_NAME', template='plotly_white', labels={'EMPLOYMENT_TYPE_NAME': 'Employment Type', 'SALARY_FROM': 'Salary ($)'})
fig.update_layout(title_font_size=20, showlegend=False)
fig.write_image('_output/salary_by_employment.png')
fig.show()
print('Full-time jobs tend to offer higher salaries compared to part-time positions. There is significant salary variation within each employment type, indicating diverse compensation structures.')

## Section 3: Salary Distribution by Industry

In [ ]:
df_industry = df.filter(df.SALARY_FROM > 0).select('NAICS2_NAME', 'SALARY_FROM').toPandas()
fig2 = px.box(df_industry, x='NAICS2_NAME', y='SALARY_FROM', title='Salary Distribution by Industry', color='NAICS2_NAME', template='plotly_white', labels={'NAICS2_NAME': 'Industry', 'SALARY_FROM': 'Salary ($)'})
fig2.update_layout(title_font_size=20, xaxis_tickangle=-45, showlegend=False)
fig2.write_image('_output/salary_by_industry.png')
fig2.show()
print('Technology and finance industries show the highest median salaries. Industries like retail and hospitality show lower salary ranges with less variation.')

## Section 4: Job Posting Trends Over Time

In [ ]:
from pyspark.sql.functions import count
df_time = df.groupBy('POSTED').agg(count('*').alias('job_count')).orderBy('POSTED').toPandas()
fig3 = px.line(df_time, x='POSTED', y='job_count', title='Job Posting Trends Over Time', template='plotly_white', labels={'POSTED': 'Date', 'job_count': 'Number of Job Postings'})
fig3.update_traces(line_color='#ec7424')
fig3.update_layout(title_font_size=20)
fig3.write_image('_output/job_trends.png')
fig3.show()
print('Job postings show seasonal fluctuations with peaks during certain months. The trend indicates growing demand for talent in the analyzed period.')

## Section 5: Top 10 Job Titles by Count

In [ ]:
df_titles = df.groupBy('TITLE_NAME').agg(count('*').alias('job_count')).orderBy('job_count', ascending=False).limit(10).toPandas()
fig4 = px.bar(df_titles, x='TITLE_NAME', y='job_count', title='Top 10 Job Titles by Count', color='job_count', template='plotly_white', labels={'TITLE_NAME': 'Job Title', 'job_count': 'Job Count'})
fig4.update_layout(title_font_size=20, xaxis_tickangle=-45)
fig4.write_image('_output/top_job_titles.png')
fig4.show()
print('Data Analyst and Software Engineer are among the most frequently posted job titles. This reflects the high demand for data and technology professionals in the current job market.')

## Section 6: Remote vs On-Site Job Postings

In [ ]:
df_remote = df.groupBy('REMOTE_TYPE_NAME').agg(count('*').alias('job_count')).toPandas()
fig5 = px.pie(df_remote, names='REMOTE_TYPE_NAME', values='job_count', title='Remote vs On-Site Job Postings', template='plotly_white', color_discrete_sequence=px.colors.qualitative.Set2)
fig5.update_layout(title_font_size=20)
fig5.write_image('_output/remote_vs_onsite.png')
fig5.show()
print('On-site jobs still dominate the job market, but remote positions represent a significant portion. The rise of remote work reflects changing workplace preferences post-pandemic.')

## Section 7: Skill Demand Analysis by Industry

In [ ]:
from pyspark.sql.functions import explode, split, trim
df_skills = df.select('NAICS2_NAME', explode(split(df.SKILLS_NAME, ',')).alias('skill')).withColumn('skill', trim(col('skill'))).filter(col('skill') != '')
from pyspark.sql.functions import col
df_skills_top = df_skills.groupBy('NAICS2_NAME', 'skill').agg(count('*').alias('skill_count')).orderBy('skill_count', ascending=False).limit(50).toPandas()
fig6 = px.bar(df_skills_top, x='NAICS2_NAME', y='skill_count', color='skill', title='Skill Demand by Industry', template='plotly_white', labels={'NAICS2_NAME': 'Industry', 'skill_count': 'Skill Count'})
fig6.update_layout(title_font_size=20, xaxis_tickangle=-45)
fig6.write_image('_output/skill_demand.png')
fig6.show()
print('Technology and professional services industries demand the most diverse skill sets. SQL, Python, and communication skills appear consistently across multiple industries.')

## Section 8: Salary Analysis by ONET Occupation Type

In [ ]:
from pyspark.sql.functions import percentile_approx
df_onet = df.filter(df.SALARY > 0).groupBy('ONET_NAME').agg(percentile_approx('SALARY', 0.5).alias('median_salary'), count('*').alias('job_count')).orderBy('median_salary', ascending=False).limit(20).toPandas()
fig7 = px.scatter(df_onet, x='ONET_NAME', y='median_salary', size='job_count', title='Salary Analysis by ONET Occupation Type', template='plotly_white', color='median_salary', labels={'ONET_NAME': 'Occupation', 'median_salary': 'Median Salary ($)'})
fig7.update_layout(title_font_size=20, xaxis_tickangle=-45)
fig7.write_image('_output/onet_salary.png')
fig7.show()
print('Senior and specialized occupations in technology show the highest median salaries. Bubble size indicates job volume, showing that high-paying roles also have significant market demand.')

## Section 9: Career Pathway Trends (Sankey Diagram)

In [ ]:
import plotly.graph_objects as go
df_sankey = df.select('SOC_2021_2_NAME', 'SOC_2021_3_NAME').filter(col('SOC_2021_2_NAME').isNotNull() & col('SOC_2021_3_NAME').isNotNull()).groupBy('SOC_2021_2_NAME', 'SOC_2021_3_NAME').agg(count('*').alias('transitions')).orderBy('transitions', ascending=False).limit(20).toPandas()
all_nodes = list(set(df_sankey['SOC_2021_2_NAME'].tolist() + df_sankey['SOC_2021_3_NAME'].tolist()))
node_idx = {n: i for i, n in enumerate(all_nodes)}
fig8 = go.Figure(go.Sankey(
    node=dict(label=all_nodes, color='#ec7424'),
    link=dict(
        source=[node_idx[s] for s in df_sankey['SOC_2021_2_NAME']],
        target=[node_idx[t] for t in df_sankey['SOC_2021_3_NAME']],
        value=df_sankey['transitions'].tolist()
    )
))
fig8.update_layout(title='Career Pathway Trends', title_font_size=20)
fig8.write_image('_output/career_pathways.png')
fig8.show()
print('Computer and mathematical occupations show the most diverse career pathways. Many professionals transition between data science and software engineering roles.')